In [218]:
from pathlib import Path as path
import shutil
import json
WORK_DIR = path(".")
DATA_DIR = WORK_DIR / "dataset"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

LAST_NAME = "Bacor"
STUDENT_ID = "TUPM-26-1879"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)




In [219]:
# [APPENDIX 1]

csv_path = DATA_DIR / f"weblog.csv"

records_processed_range = 1000 + ID_SUM 
target_status_class = (SEED_DIGIT % 4) + 2

record = []
missing_records = 0
invalid_records = 0
rows_examined = 0
with open(csv_path, "r", encoding="utf-8") as file:
    header = file.readline().strip().split(",")
    for _ in range(records_processed_range):
        line = file.readline()

        if not line:
            break

        rows_examined += 1
        
        if not line.strip():
            missing_records += 1
            continue

        value = line.strip().split(",")

        if len(value) != len(header):
            invalid_records += 1
            continue

        url = value[2].strip()
        status = value[3].strip()

       
        if not url or not status:
            missing_records += 1
            continue

        if len(status) != 3 or not status.isdigit() or not 100 <= int(status) <= 599:
            invalid_records += 1
            continue

        record.append([field.strip() for field in value])

        


target_list = []
status_code = []
seen_status = []
status_count = {}
for value in record:
    status = value[3].strip()
    if status.startswith(str(target_status_class)):
        target_list.append(value) 
        # if-else status code generated by: ChatGPT, prompt: "how can i not explicitly state the status codes?"
        if status in status_code:
            seen_status.append(value[3])
            status_count[status] += 1

        else:
            status_code.append(value[3])
            status_count[status] = 1

rrl = []
seen_rrls = []
rrl_count = {}
for value in target_list:
    url = value[2].strip()
    if url in seen_rrls:
        rrl.append(url)
        rrl_count[url] += 1
    else:
        seen_rrls.append(url)        
        rrl_count[url] = 1

sorted_rrls = sorted(
    rrl_count.items(),
    key = lambda x: x[1],
    reverse = True
)[:5]

csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_webanalysis.csv"

with open(csv_path, "w", encoding="utf-8") as file:
    file.write(f"{','.join(header)}\n")
    # for loop generated by: ChatGPT, prompt: "how can i do it with the standard write function"
    for value in target_list:
        file.write(f"{','.join(value)}\n")

summary_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_webanalysis_summary.json"
summary_data = {
    "ID Sum": ID_SUM,
    "Seed Digit": SEED_DIGIT,
    "Processed Records Row Count": rows_examined,
    "Valid Records Row Count": f"{len(record)}",
    "Inalid Records Row Count": invalid_records,
    "Missing Records Row Count": missing_records,
    "Target Status Class": f"{target_status_class}xx",
    "Target Class Row Count": f"{len(target_list)}",
    "Counts per Status Code": {
        # generated by ChatGPT, prompt: "how can i insert the status and count for each status code"
        status: count
        for status, count in status_count.items()
    },
    "Overall Status Code Count": sum(status_count.values()),
    "Filtered CSV File": csv_path.name,
    "JSON Summary File": summary_path.name
}
with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary_data, file, indent=4)

print("Created:", summary_path)

backup_path = BACKUP_DIR / summary_path.name

shutil.copy2(summary_path, backup_path)

print("Created:", backup_path)


# PRINTING


hder = f"{header[0]:^10} | {header[1]:^35} | {header[2]:^80} | {header[3]:^7}"
width = len(hder)
print("-" * width)
print(f"{"RECORDS PROCESSED":^{width}}")
print("-" * width)
print(hder)
print("-" * width)
for values in record:
    print(f"{values[0]:<10} | {values[1]:<35} | {values[2]:<80} | {values[3]:<7}")
print("-" * width)
print()


hder = f"{header[0]:^10} | {header[1]:^35} | {header[2]:^80} | {header[3]:^7}"
width = len(hder)
print("-" * width)
print(f"{"RECORDS IN TARGET CLASS":^{width}}")
print("-" * width)
print(hder)
print("-" * width)
for values in target_list:
    print(f"{values[0]:<10} | {values[1]:<35} | {values[2]:<80} | {values[3]:<7}")
print("-" * width)
print()

print("-" * 50)
print(f"COUNT PER STATUS CODE")
for status, count in status_count.items():
    print(f"Status Code {status}: {count} records")
print("Overall Status Count:", sum(status_count.values()))
print("-" * 50)
print()

print("-" * 50)
print(f"FIVE MOST REQUESTED URLs")
for rrl, count in sorted_rrls:
    print(f"{rrl}: {count} counts")
print("-" * 50)
print()

print(f"Target Status Class: {target_status_class}xx")
print(f"JSON Summary File: {summary_path}")
print(f"JSON Summary Path Exists: {summary_path.exists()}")
print(f"Backup Summary Path Exists: {backup_path.exists()}")



    
 




Created: output\Bacor_TUPM-26-1879_webanalysis_summary.json
Created: backup\Bacor_TUPM-26-1879_webanalysis_summary.json
---------------------------------------------------------------------------------------------------------------------------------------------
                                                              RECORDS PROCESSED                                                              
---------------------------------------------------------------------------------------------------------------------------------------------
    IP     |                Time                 |                                       URL                                        |  Staus 
---------------------------------------------------------------------------------------------------------------------------------------------
10.128.2.1 | [29/Nov/2017:06:58:55               | GET /login.php HTTP/1.1                                                          | 200    
10.128.2.1 | [29/Nov/2017:06

In [220]:
# [APPENDIX 2]

data_path = DATA_DIR / f"user_behavior_dataset.csv"

operating_system = ("Android" if (SEED_DIGIT % 2) == 0 else "iOS")
user_behavior_class = (SEED_DIGIT % 5) + 1

filtered_records = []
with open(data_path, "r", encoding="utf-8") as file:
    # generated by ChatGPT, prompt: "in the uploaded csv and the current code snippet, how can i calculate the average battery drain and data usage of users while handling cases with no matching records"
    header = file.readline().strip().split(",")
    user_id_index = header.index("User ID")
    device_index = header.index("Device Model")
    os_index = header.index("Operating System")
    class_index = header.index("User Behavior Class")
    battery_index = header.index("Battery Drain (mAh/day)")
    data_index = header.index("Data Usage (MB/day)")
    for line in file:
        if not line.strip():
            continue

        row = line.strip().split(",")

        # Ignore rows that do not contain all expected columns
        if len(row) != len(header):
            continue

        if (
            row[os_index] == operating_system
            and row[battery_index]
            and row[data_index]
        ):
            filtered_records.append(row)

selected_records = []
for value in filtered_records:
    if int(value[class_index]) == user_behavior_class:
        selected_records.append(value)

if not selected_records:
    print("No mathing records")
else:
    for value in selected_records:
        battery_values = []
        data_values = []

        battery = float(row[battery_index])
        data = float(row[data_index])

        battery_values.append(battery)
        data_values.append(data)

        if battery_values and data_values:
            average_battery_drain = sum(battery_values) / len(battery_values)
            average_data_usage = sum(data_values) / len(data_values)

exported_header = [
            value[user_id_index],
            value[device_index],
            value[os_index],
            value[battery_index],
            value[data_index]
        ]
csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_usageprofile.csv"
with open(csv_path, "w", encoding="utf-8") as file:
    file.write("User ID,Device Model,Operating System,Battery Drain (mAh/day),Data Usage (MB/day)\n")
    for value in selected_records:
        file.write(f"{','.join(exported_header)}\n") # this cant happen because the fields are not equal
print("Created:", csv_path)

with open(csv_path, "r", encoding="utf-8") as file:
    exported_records = file.readlines()


summary_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_usageprofile_summary.json"
summary_data = {
    "Seed Digit": SEED_DIGIT,
    "Operating System": operating_system,
    "User Behavior Class": user_behavior_class,
    "Selected Row Count": f"{len(selected_records)}",
    "Average Battery Drain (mAh/day)": f"{average_battery_drain} mAh/day",
    "Average Data Usage (MB/day)": f"{average_data_usage} MB/day",
    "Exported CSV File": csv_path.name,
    "Exported Row Count": f"{len(exported_records) - 1}", # reduce 1 due to the header not being included in selected row count
    "Count Verification": ("PASSED" if (len(exported_records) - 1) == len(selected_records) else "FAILED"),
    "JSON Summary File": summary_path.name
}

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary_data, file, indent=4)

print("Created:", summary_path)

backup_path = BACKUP_DIR / summary_path.name

shutil.copy2(summary_path, backup_path)

print("Created:", backup_path)
file_info = backup_path.stat()
# PRINTING
## column formatting generated by ChatGPT
column_widths = [10, 15, 16, 30, 35, 25, 35, 20, 7, 12, 7, 20]

hder = " | ".join(
    f"{name:^{column_widths[i]}}"
    for i, name in enumerate(header)
)
width = len(hder)
print("-" * width)
print(f"{'RECORDS MATCHING OPERATING SYSTEM, Count: ' + str(len(filtered_records)):^{width}}")
print("-" * width)
print(hder)
print("-" * width)
for values in filtered_records:
    print(" | ".join(
        f"{value:<{column_widths[i]}}"
        for i, value in enumerate(values)
    ))
print("-" * width)
print()


column_widths = [10, 15, 16, 30, 35, 25, 35, 20, 7, 12, 7, 20]

hder = " | ".join(
    f"{name:^{column_widths[i]}}"
    for i, name in enumerate(header)
)
width = len(hder)
print("-" * width)
print(f"{'RECORDS IN TARGET BEHAVIOR CLASS, Count: ' + str(len(selected_records)):^{width}}")
print("-" * width)
print(hder)
print("-" * width)
for values in selected_records:
    print(" | ".join(
        f"{value:<{column_widths[i]}}"
        for i, value in enumerate(values)
    ))
print("-" * width)
print()

print()
print("-" * 50)
print(f"{'MOBILE DEVICE USAGE PROFILING PERFORMANCE REPORT':^50}")
print("-" * 50)
print(f"Operating System: {operating_system}")
print(f"Behavior Class: {user_behavior_class}")
print(f"Average Battery Drain: {average_battery_drain} mAh/day")
print(f"Average Data Usage: {average_data_usage} MB/day")
print(f"Exported CSV File: {csv_path.name}")
print(f"Count Verification: {"PASSED" if (len(exported_records) - 1) == len(selected_records) else "FAILED"}")
print(f"JSON Summary File: {summary_path.name}")
print(f"Backup File & Size: {backup_path.name} with {file_info.st_size} bytes")
print("-" * 50)



Created: output\Bacor_TUPM-26-1879_usageprofile.csv
Created: output\Bacor_TUPM-26-1879_usageprofile_summary.json
Created: backup\Bacor_TUPM-26-1879_usageprofile_summary.json
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
                                                                                                        RECORDS MATCHING OPERATING SYSTEM, Count: 146                                                                                                         
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 User ID   |  Device Model   | Operating System |    App Usag

In [ ]:
# [APPENDIX 3] 

target_year = 2015 + SEED_DIGIT # Originally: 2015 + 9 =
csv_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"

valid_incidents = []
rows_examined = 0
missing_val = 0
invalid_val = 0
with open(csv_path, "r", encoding="utf-8") as file:
    header = file.readline().strip().split(",")
    for line in file:
        if not line:
            break

        rows_examined += 1

        if not line.strip():
            missing_val += 1
            continue

        value = line.strip().split(",")

        if len(value) != len(header):
            invalid_val += 1
            continue
        
        year = value[1].strip()

        if not year:
            missing_val += 1
            continue

        if (
            int(year) == target_year and year.isnumeric() and 2015 <= int(year) <= 2024
        ):
            valid_incidents.append(value)

attack_type = []
seen_type = []
type_count = {}
for value in valid_incidents:
    attack = value[2].strip()        
    if attack in attack_type:
        seen_type.append(value[2])
        type_count[attack] += 1

    else:
        attack_type.append(value[2])
        type_count[attack] = 1

print(f"COUNT PER ATTACK TYPE")
for attack, count in type_count.items():
    print(f"{attack}: {count} counts")
print()
costliest_incidents = sorted(
    valid_incidents,
    key = lambda x: float(x[4].strip()),
    reverse=True
)

# financial losses generated by ChatGPT, prompt: "how can i check for the total and average financial loss of the whole valid_incidents"
financial_losses = [
    float(value[4].strip())
    for value in valid_incidents
    if value[4].strip()
]
total_financial_losses = sum(financial_losses)
average_financial_loss = total_financial_losses / len(valid_incidents)

json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_costliestincidents.json"

# ranking incudebts generated by ChatGPT, prompt: "how can i retain the original dataset if the losses are tied in ranking"
ranked_incidents = sorted(
    enumerate(valid_incidents),
    key=lambda item: float(item[1][4].strip()),
    reverse=True
)

if len(ranked_incidents) > 5:
    fifth_loss = float(ranked_incidents[4][1][4].strip())

    costliest_incidents = [
        value
        for original_index, value in ranked_incidents
        if float(value[4].strip()) >= fifth_loss
    ]
else:
    costliest_incidents = [
        value for original_index, value in ranked_incidents
    ]

# Restore original dataset order among the selected incidents
original_order = {id(value): i for i, value in enumerate(valid_incidents)}
costliest_incidents.sort(key=lambda value: original_order[id(value)])

costliest_records = [
    dict(zip(header, incident))
    for incident in costliest_incidents
]
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(costliest_records, file, indent=4)

print("Created:", json_path.name)

csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_filteredincidents.csv"
with open(csv_path, "w", encoding="utf-8") as file:
    file.write(f"{','.join(header)}\n")
    for values in valid_incidents:
        file.write(f'{','.join(values)}\n')

backup_path = BACKUP_DIR / json_path.name
print("Created:", csv_path.name)

shutil.copy2(json_path, backup_path)

print("Created:", backup_path)
print(f"Backup Path Exists: {backup_path.exists()}")
print(f"Total Financial Loss: {total_financial_losses}")
print(f"Average Financial Losses: {round(average_financial_loss, 3)}\n")
print(f"Filtered CSV File: {csv_path}\n")
print(f"JSON Summary File: {json_path}\n")
print(f"Backup File: {json_path}\n")

report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_reportincidents.txt"
with open(report_path, "w", encoding="utf-8") as file:
    file.write("CYBERSECURITY INCIDENT REPORT:\n")
    file.write(f"----------------------------------\n")    
    file.write(f"Target Year: {target_year}\n")
    file.write(f"Dataset Incident Count: {rows_examined}\n")
    file.write(f"Target Year Incident Count: {len(valid_incidents)}\n")
    file.write(f"Total Financial Losses: ${total_financial_losses}M\n")
    file.write(f"Average Financial Losses: ${round(average_financial_loss, 3)}M\n")
    file.write(f"Five Costliest Incidents\n")
    for rank, value in enumerate(costliest_incidents, start=1):
        file.write(
            f"{rank}: "
            f"Country: {value[0]:<10} | "
            f"Attack Type: {value[2]:<15} | "
            f"Financial Loss: ${float(value[4]):,.2f}M"
            f"\n"
        )
    file.write(f"Filtered CSV File: {csv_path}\n")
    file.write(f"JSON Summary File: {json_path}\n")
    file.write(f"Backup File: {json_path}\n")
    file.write(f"Text Report File: {report_path}\n")
print("Report Created:", report_path)



COUNT PER ATTACK TYPE
Ransomware: 53 counts
DDoS: 54 counts
Phishing: 62 counts
Malware: 43 counts
Man-in-the-Middle: 47 counts
SQL Injection: 40 counts

Created: Bacor_TUPM-26-1879_costliestincidents.json
Created: Bacor_TUPM-26-1879_filteredincidents.csv
Created: backup\Bacor_TUPM-26-1879_costliestincidents.json
Backup Path Exists: True
Total Financial Loss: 15434.29
Average Financial Losses: 51.62

Filtered CSV File: output\Bacor_TUPM-26-1879_filteredincidents.csv

JSON Summary File: output\Bacor_TUPM-26-1879_costliestincidents.json

Backup File: output\Bacor_TUPM-26-1879_costliestincidents.json

Report Created: output\Bacor_TUPM-26-1879_reportincidents.txt
